In [4]:
!git clone https://github.com/meta-compute/sem-fracture-deep-learning.git

Cloning into 'sem-fracture-deep-learning'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 2691 (delta 0), reused 3 (delta 0), pack-reused 2685 (from 2)
Receiving objects: 100% (2691/2691), 904.02 MiB | 40.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (2000/2000), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [10]:
cd sem-fracture-deep-learning

/kaggle/working/sem-fracture-deep-learning


In [12]:
!git checkout vishnu-dev
!pip install -r requirements.txt

Updating files: 100% (635/635), done.
branch 'vishnu-dev' set up to track 'origin/vishnu-dev'.
Switched to a new branch 'vishnu-dev'


# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [5]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "tinyvit_21m"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m
MODE = "finetune"  # head_only or finetune
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

INITIAL_CHECKPOINT = BASELINE_CHECKPOINTS[MODEL_KEY] if MODE == "finetune" else None
BACKBONE_LR = 1e-5 if MODE == "finetune" else None
HEAD_LR = 1e-4 if MODE == "finetune" else 1e-3
EPOCHS = 30 if MODEL_KEY == "tinyvit_21m" and MODE == "finetune" else 10
PATIENCE = 4 if MODE == "finetune" else None
GRAD_CLIP_NORM = 1.0 if MODEL_KEY.startswith("tinyvit") and MODE == "finetune" else None
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"/"loss_mod"/ f"{MODEL_KEY}"/f"{MODEL_KEY}_{MODE}_best.pth"
)
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [6]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    loss_type="class_balanced",
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=False,
    seed=42,
    batch_size=16,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])


Epoch 1/30 starting
Epoch 1/30 | train_loss=0.2225 | val_loss=0.0601 | val_accuracy=0.9786 | val_macro_f1=0.9451 | elapsed=27.3s | best_checkpoint=True
Epoch 2/30 starting
Epoch 2/30 | train_loss=0.2032 | val_loss=0.0572 | val_accuracy=0.9786 | val_macro_f1=0.9470 | elapsed=20.9s | best_checkpoint=True
Epoch 3/30 starting
Epoch 3/30 | train_loss=0.1912 | val_loss=0.0655 | val_accuracy=0.9786 | val_macro_f1=0.9375 | elapsed=20.6s | best_checkpoint=False
Epoch 4/30 starting
Epoch 4/30 | train_loss=0.1690 | val_loss=0.0877 | val_accuracy=0.9714 | val_macro_f1=0.9201 | elapsed=20.7s | best_checkpoint=False
Epoch 5/30 starting
Epoch 5/30 | train_loss=0.1541 | val_loss=0.1600 | val_accuracy=0.9643 | val_macro_f1=0.8954 | elapsed=20.8s | best_checkpoint=False
Epoch 6/30 starting
Epoch 6/30 | train_loss=0.2873 | val_loss=0.0685 | val_accuracy=0.9821 | val_macro_f1=0.9515 | elapsed=20.9s | best_checkpoint=True
Epoch 7/30 starting
Epoch 7/30 | train_loss=0.1566 | val_loss=0.0741 | val_accuracy=0

## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [14]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split.csv",
    device=DEVICE,
    batch_size=16,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])


{'confusion_matrix': [[117, 0, 0], [0, 142, 0], [0, 4, 17]], 'accuracy': 0.9857142857142858, 'precision': [1.0, 0.9726027397260274, 1.0], 'recall': [1.0, 1.0, 0.8095238095238095], 'class_f1': [1.0, 0.9861111111111112, 0.8947368421052632], 'macro_f1': 0.9602826510721248}


,filename,actual,predicted,correct
0,100Cr6_AR_06_10.png,Brittle,Brittle,True
1,100Cr6_AR_06_3.png,Brittle,Brittle,True
2,304_H2_02_1.png,Brittle,Brittle,True
3,304_H2_02_10.png,Brittle,Brittle,True
4,304_H2_02_11.png,Brittle,Brittle,True
...,...,...,...,...
275,mX65_H2_29_22.png,Mixed,Mixed,True
276,mX65_H2_29_23.png,Mixed,Mixed,True
277,mX65_H2_29_3.png,Mixed,Mixed,True
278,mX65_H2_29_31.png,Mixed,Mixed,True


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [9]:
RUN_FINAL_TEST_EVALUATION = True

In [13]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split.csv",
        device=DEVICE,
        batch_size=16,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")


Final held-out test metrics: {'confusion_matrix': [[116, 0, 0], [0, 141, 2], [4, 3, 13]], 'accuracy': 0.967741935483871, 'precision': [0.9666666666666667, 0.9791666666666666, 0.8666666666666667], 'recall': [1.0, 0.986013986013986, 0.65], 'class_f1': [0.983050847457627, 0.9825783972125435, 0.7428571428571429], 'macro_f1': 0.9028287958424378}
